In [24]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18 # Standard ResNet18 architecture
import numpy as np # Used for numerical operations
from scipy.optimize import differential_evolution # DE optimizer used to search over pixel coordinates and RGB values

# Check if we have a GPU available, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [25]:
""" Image Preprocessing - CIFAR-10 normalization values """

# CIFAR-10 channel-wise normalization (RGB)
# Used to normalize images so the model sees standardized input
# Values gotten from https://github.com/huyvnphan/PyTorch_CIFAR10
mean_vals = [0.4914, 0.4822, 0.4465]
std_vals = [0.2023, 0.1994, 0.2010] 

# Convert the mean and std values into tensors of shape 3, 1 ,1 (RGB, x-axis, y-axis)
# Each pixel will be normalized in their respective RGB channel using these tensors
mean = torch.tensor(mean_vals).view(3, 1, 1)
std = torch.tensor(std_vals).view(3, 1, 1)

# Simple transform. Converts image to PyTorch tensor. Scales image (CHW) to float pixels between 0.0 and 1.0
transform_raw = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
])

# Load CIFAR-10 test dataset
# Train = False since we are only attacking, not training
# Download = True will download the dataset if not already on local machine
# Transform defines the transform used on the images in the dataset
testset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_raw
)

# Create data loader with batch size 1 for single image processing
# Effectively wraps the testset in the loader where we can batch them and shuffle them
testloader = torch.utils.data.DataLoader(testset, batch_size=1, shuffle=True)

# Helper to build network with CIFAR-10 compatible ResNet-18
def make_cifar_resnet18():
    # Start with standard ResNet-18 architecture without weights
    model = torchvision.models.resnet18(weights=None)
    
    # Modify first conv layer for smaller input (3x32x32)
    # Basically converts the input layer to another format since ResNet18 is designed for 224x224
    # Makes sure the network doesnt change the resolution of the image since we need all pixel data for the attack
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    
    # Adjust max pooling to avoid over-downsampling on small images
    model.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
    
    # Replace the original 1000 class ImageNet classifer with a 10-class
    model.fc = nn.Linear(512, 10)

    # Load pretrained weights 
    state_dict = torch.load("resnet18.pt", map_location="cpu")
    model.load_state_dict(state_dict)
    
    return model

# Load model and prepare for inference
model = make_cifar_resnet18()
model.to(device).eval();

In [26]:
# CIFAR-10 class labels in order (0-9)
CIFAR10_CLASSES = [ "airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

@torch.no_grad() # No gradient (only used for back propagation in training)
def predict(model, img_norm):
    """Returns prediction and confidence for a normalized image."""
    # Forward pass through model
    # Add a batch dimension even though we only process one im age (PyTorch expects that)
    # Saves the raw scores / logits
    logits = model(img_norm.unsqueeze(0))
    
    # Convert logits to probabilities
    probs = torch.softmax(logits, dim=1)[0]
    
    # Get class with highest probability
    pred = probs.argmax().item()
    
    # Get confidence score for predicted class
    conf = probs[pred].item()
    return pred, conf

def get_fitness_function(img_norm, label, model, mean, std, n_pixels=1):
    """
    Creates and returns a fitness function for the n-pixel attack.
    
    Based on Vargas & Kouichi (2019) one-pixel attack: https://arxiv.org/pdf/1710.08864
    
    Args:
        img_norm: Normalized image tensor [C, H, W]
        label: Actual label of image
        model: Model to attack
        mean, std: Normalization statistics
        n_pixels: Number of pixels to perturb
    """
    
    def fitness_and_pred(candidate):
        # Create adversary variable - a copy of the original image
        adv = img_norm.clone()
        
        # Modify each target pixel
        for i in range(n_pixels):
            # Candidate vector: [x, y, r, g, b] for each pixel
            # The Candidate holds n x 5 amount of values.
            # E.g. for 1 pixel attack its 5 values that are changed
            # 2 pixel attack is 10 values etcetera
            idx = i * 5
            x_pos = int(round(candidate[idx]))
            y_pos = int(round(candidate[idx + 1]))
            
            # Extract RGB values from optimizer (in [0, 255] range). 
            # 2, 3, 4 comes from their index within that pixel (x, y, r, g, b)
            r_255 = candidate[idx + 2]
            g_255 = candidate[idx + 3]
            b_255 = candidate[idx + 4]
            
            # Convert RGB from [0, 255] to normalized space using CIFAR-10 stats
            r_norm = (r_255 / 255.0 - mean[0]) / std[0]
            g_norm = (g_255 / 255.0 - mean[1]) / std[1]
            b_norm = (b_255 / 255.0 - mean[2]) / std[2]
            
            # Set pixel values in adversarial image
            adv[0, y_pos, x_pos] = r_norm
            adv[1, y_pos, x_pos] = g_norm
            adv[2, y_pos, x_pos] = b_norm
        
        # Evaluate modified image just like we do in predict
        with torch.no_grad():
            logits = model(adv.unsqueeze(0))
            probs = torch.softmax(logits, dim=1)[0]

        pred = probs.argmax().item() # Predicted label index
        conf = probs[pred].item()    # Confidence in the predicted label
        fit = probs[label].item()    # How confident is model in true class
        
        return fit, pred, conf, adv
    
    return fitness_and_pred

In [27]:
def differential_evolution_attack_n_pixels(img_norm, label, model, mean, std,
                                          n_pixels=1, popsize=80, maxiter=60):
    """
    Runs n-pixel attack using differential evolutionl.
    
    Searches for n pixel perturbations that fool the model.
    Based on Vargas & Kouichi (2019) one-pixel attack https://arxiv.org/pdf/1710.08864
    
    Arguments:
        img_norm: Normalized image tensor [C, H, W]
        label: actual class label
        model: model to attack
        mean, std: CIFAR-10 normalization statistics
        n_pixels: Number of pixels to alter/perturb
        popsize: Population size for DE optimizer
        maxiter: Maximum generations to run
    
    Returns:
        pixels: List of (x, y, r, g, b) tuples for modified pixels
        iterations: Actual number of generations used in the attack
        final_fitness: Probability of correct class label after attack
        final_pred: Model's predicted label on adversarial image
        final_conf: Confidence in adversial class label
    """
    # Define search bounds for each pixel: (x, y, R, G, B)
    pixel_bounds = [
        (0, 31),   # x coordinate (CIFAR-10 is 32x32)
        (0, 31),   # y coordinate
        (0, 255),  # Red channel value
        (0, 255),  # Green channel value
        (0, 255),  # Blue channel value
    ]

    # Creates n tuples, one for each "attack pixel"
    bounds = pixel_bounds * n_pixels

    # Builds a fitness function that gets fitness, prediction, confidence and adversial pixel data
    fitness_and_pred = get_fitness_function(
        img_norm, label, model, mean, std, n_pixels
    )
    
    # Track optimization progress
    generation = [0]   # Amount of generations currently 
    best_fitness = [float('inf')] # Best fitness value (float)
    best_candidate = [None] # Best attack candidate
    
    def wrapped_fitness(c):
        """
        Fitness wrapper to track best candidate.
          Makes it possible to access the fitness function.
          calculates fitness of c and if it is better than best candidate, update and return that candidate
        """
        fit = fitness_and_pred(c)[0]
        if fit < best_fitness[0]:
            best_fitness[0] = fit
            best_candidate[0] = c.copy()
        return fit
    
    def callback(xk, convergence):
        """
        Called after each DE generation has finished
        It will stop early if we have fooled the model to believe another label is true!
        """
        generation[0] += 1
        
        # Stop if we've successfully fooled the model
        if best_candidate[0] is not None:
            _, pred, _, _ = fitness_and_pred(best_candidate[0])
            if pred != label:
                return True  # Attack succeeded, no need to continue
        
        return False
    
    # Run differential evolution optimizer
    # With settings similar to the One Pixel Attack Article
    result = differential_evolution(
        wrapped_fitness,
        bounds,
        strategy="best1bin",
        maxiter=maxiter,
        popsize=popsize,
        mutation=(0.5, 1),
        recombination=1,
        callback=callback,
        atol=-1,
        tol=0.01,
        polish=False,
    )
    
    # Evaluate final adversarial image
    final_fit, final_pred, final_conf, adv_img = fitness_and_pred(result.x)
    
    # Parse pixel coordinates and RGB values from solution
    # Build a list of the coordinate + RGB values of each modified pixel
    pixels = []
    for i in range(n_pixels):
        idx = i * 5
        x = int(round(result.x[idx]))
        y = int(round(result.x[idx + 1]))
        r = result.x[idx + 2]
        g = result.x[idx + 3]
        b = result.x[idx + 4]
        pixels.append((x, y, r, g, b))
    
    return pixels, result.nit, final_fit, final_pred, final_conf

def attack_image_n_pixels(model, img_norm, true_label, mean, std, 
                          n_pixels=1, popsize=80, maxiter=60):
    """
    Wrapper function so we can do the attack from outside this object
    Runs n-pixel attack and return whether it succeeded.
    """
    
    # Run the differential evolution attack with specified parameters
    pixels, generations, final_fitness, adv_pred, adv_conf = differential_evolution_attack_n_pixels(
        img_norm, true_label, model, mean, std, 
        n_pixels=n_pixels, popsize=popsize, maxiter=maxiter
    )
    
    # Check if attack fooled the model (prediction changed)
    success = (adv_pred != true_label)
    
    return success, adv_pred, adv_conf, generations, final_fitness, pixels

In [28]:
# Shared helper functions
@torch.no_grad()
def get_baseline_prediction(model, img_norm, true_label):
    """Get the model's prediction and confidence for the true class label of the image."""
    
    # Get predicted class and its confidence
    pred, conf = predict(model, img_norm)
    
    # Get probability distribution over the 10 CIFAR-10 labels
    probs = torch.softmax(model(img_norm.unsqueeze(0)), dim=1)[0]
    
    # Extract probability for the actual / true class of the image
    true_class_prob = probs[true_label].item()
    
    return pred, conf, true_class_prob

def normalize_image(img, mean, std):
    """Apply CIFAR-10 normalization to image."""
    return (img - mean.to(img.device)) / std.to(img.device)

In [29]:
def print_attack_start(image_num, true_label, pred, conf, true_class_prob):
    """Print header info before attacking an image."""
    true_name = CIFAR10_CLASSES[true_label]
    pred_name = CIFAR10_CLASSES[pred]
    print(f"\nAttacking image {image_num} | true = {true_name} | pred = {pred_name} ({conf:.3f})")
    print(f"Initial prob of TRUE class ({true_name}): {true_class_prob:.4f}")

def print_attack_result(success, true_label, adv_pred, adv_conf, pixels=None):
    """Print a short summary of the attack, with optional pixel details."""
    true_name = CIFAR10_CLASSES[true_label]
    adv_name = CIFAR10_CLASSES[adv_pred]

    if success:
        print(f"Attack succeeded: {true_name} → {adv_name} (confidence {adv_conf:.3f})")
    else:
        print(f"Attack failed: still {true_name} (confidence {adv_conf:.3f})")

def print_evaluation_summary(tried, success_count, total_gens_success):
    """Print final statistics after evaluation."""
    rate = (success_count / tried * 100) if tried > 0 else 0.0
    avg_gens = total_gens_success / success_count if success_count > 0 else 0.0
    
    print(f"\n{'='*60}")
    print(f"Finished {tried} correctly classified images.")
    print(f"Success rate: {rate:.2f}%")
    print(f"Average generations (successful attacks): {avg_gens:.2f}")
    print(f"{'='*60}\n")
    
    return rate

# Per-class evaluation helpers
def initialize_class_stats():
    """Initialize statistics tracking for all CIFAR-10 classes."""
    class_stats = {}
    for class_idx, class_name in enumerate(CIFAR10_CLASSES):
        class_stats[class_idx] = {
            'name': class_name,
            'success': 0,
            'total': 0,
            'total_gens': 0
        }
    return class_stats

def print_per_class_header(n_pixels, images_per_class):
    """Print header for per-class evaluation."""
    print(f"\n{'='*70}")
    print(f"Per-Class {n_pixels}-Pixel Attack Evaluation")
    print(f"Testing {images_per_class} correctly classified images per class")
    print(f"{'='*70}\n")

def print_attack_progress(true_name, current_count, images_per_class, conf):
    """Print progress indicator for current attack."""
    print(f"[{true_name} {current_count}/{images_per_class}] conf: {conf:.3f}", end=" ")

def update_class_stats(class_stats, true_label, success, gens):
    """Update statistics after an attack attempt."""
    if success:
        class_stats[true_label]['success'] += 1
        class_stats[true_label]['total_gens'] += gens

def print_per_class_summary(class_stats, n_pixels):
    """Print detailed per-class statistics table."""
    print(f"\n{'='*70}")
    print(f"Per-Class Attack Success Rate ({n_pixels}-pixel)")
    print(f"{'='*70}")
    print(f"{'Class':<15} {'Success':<12} {'Rate':<10} {'Avg Iters'}")
    print(f"{'-'*70}")
    
    total_success = 0
    total_tested = 0
    
    for class_idx in range(10):
        stats = class_stats[class_idx]
        name = stats['name'].capitalize()
        success = stats['success']
        total = stats['total']
        total_success += success
        total_tested += total
        
        if total > 0:
            rate = (success / total) * 100
            avg_gens = stats['total_gens'] / success if success > 0 else 0
            print(f"{name:<15} {success}/{total:<10} {rate:>5.1f}%     {avg_gens:>5.1f}")
        else:
            print(f"{name:<15} 0/0          -")
    
    print(f"{'-'*70}")
    overall_rate = (total_success / total_tested * 100) if total_tested > 0 else 0
    print(f"{'OVERALL':<15} {total_success}/{total_tested:<10} {overall_rate:>5.1f}%")
    print(f"{'='*70}\n")


In [30]:
# Main evaluation functions
@torch.no_grad()
def evaluate_attack_n_pixels(model, dataloader, mean, std, n_pixels=1, num_images=100, popsize=80, maxiter=60,):
    """
    Run n-pixel attack on test images and outputs statistics
    
    Argumentss:
        model: Target model
        dataloader: Torch data loader
        mean, std: Normalization statistics
        n_pixels: Number of pixels to attack /perturb
        num_images: Maximum images to test
        popsize: DE optimizer population size
        maxiter: DE optimizer max generations
        label_filter: If set, only attack images with this true label
    
    Return:
        Attack success rate as percentage
    """
    success_count = 0 # Total amount of successes
    tried = 0 # Total images attacked
    total_gens_success = 0 # Total generations to success for successful attacks

    # Print a header
    print(f"\n{'='*60}")
    print(f"Running {n_pixels}-PIXEL ATTACK")
    print(f"{'='*60}\n")

    # Loop through each image / label pair in the dataloader
    for images, labels in dataloader:
        img = images[0].to(device) # Get image data
        true_label = labels[0].item() # Get true label
        
        # Normalize the image and get baseline prediction values (predicted label, confidence and true class label)
        img_norm = normalize_image(img, mean, std)
        pred, conf, true_class_prob = get_baseline_prediction(model, img_norm, true_label)
        
        # Only attack correctly classified images e.g. skip images that are mislabeled by the model
        if pred != true_label:
            continue

        # Increment tried amount of images and print the start statistics of current attack
        tried += 1
        print_attack_start(tried, true_label, pred, conf, true_class_prob)
        
        # Execute attack
        success, adv_pred, adv_conf, gens, final_fit, pixels = attack_image_n_pixels(
            model, img_norm, true_label, mean, std, 
            n_pixels=n_pixels, popsize=popsize, maxiter=maxiter
        )

        # Print results of the attack
        print_attack_result(success, true_label, adv_pred, adv_conf, pixels)

        # Increment successes if it was successful, and also adds the generations to succeed
        if success:
            success_count += 1
            total_gens_success += gens
        
        if tried >= num_images:
            break

    # Returns and prints the summary of the attack
    return print_evaluation_summary(tried, success_count, total_gens_success)

@torch.no_grad()
def evaluate_attack_per_class(model, dataloader,mean, std, n_pixels=1, images_per_class=10, popsize=80, maxiter=60,):
    """
    Evaluate n-pixel attack success rate across CIFAR-10 classes.
    
    Runs the attack on a fixed number of correctly classified images per class
    and reports per-class and overall success statistics.
    
    Args:
        model: Target model to attack
        dataloader: torch DataLoader
        mean, std: Normalization statistics
        n_pixels: Number of pixels to attack/perturb
        images_per_class: Number of images to test per class / label
        popsize: DE population size
        maxiter: DE max generations
    """
    class_stats = initialize_class_stats()
    print_per_class_header(n_pixels, images_per_class)
    
    # Track which classes have collected enough samples
    classes_complete = {i: False for i in range(10)}
    
    for images, labels in dataloader:
        # Stop when we've tested enough images from all classes
        if all(classes_complete.values()):
            break
        
        img = images[0].to(device)
        true_label = labels[0].item()
        
        # Skip if we've already tested enough from this class
        if class_stats[true_label]['total'] >= images_per_class:
            classes_complete[true_label] = True
            continue
        
        # Normalize and get prediction
        img_norm = normalize_image(img, mean, std)
        pred, conf = predict(model, img_norm)
        
        # Only attack images the model classifies correctly
        if pred != true_label:
            continue
        
        true_name = CIFAR10_CLASSES[true_label]
        class_stats[true_label]['total'] += 1
        current_count = class_stats[true_label]['total']
        
        print_attack_progress(true_name, current_count, images_per_class, conf)
        
        # Execute attack
        success, adv_pred, adv_conf, gens, final_fit, pixels = attack_image_n_pixels(
            model, img_norm, true_label, mean, std, 
            n_pixels=n_pixels, popsize=popsize, maxiter=maxiter
        )
        
        print_attack_result(success, true_label, adv_pred, adv_conf)
        update_class_stats(class_stats, true_label, success, gens)
    
    print_per_class_summary(class_stats, n_pixels)
    return class_stats

In [31]:
@torch.no_grad()
def evaluate_model(model, loader, device, mean, std):
    """Evaluate model accuracy on test set"""
    model.eval()
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        # Normalize images
        images = (images - mean.to(device)) / std.to(device)
        
        logits = model(images)
        preds = logits.argmax(dim=1)
        
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = correct / total * 100
    print(f"Test accuracy: {acc:.2f}% ({correct}/{total})")
    return acc

# Evaluate model
evaluate_model(model, testloader, device, mean, std)

Test accuracy: 92.59% (9259/10000)


92.58999999999999

In [32]:
# Run 1-pixel attack
evaluate_attack_n_pixels(model, testloader, mean, std, n_pixels=1, num_images=10, popsize=80, maxiter=40)


Running 1-PIXEL ATTACK


Attacking image 1 | true = frog | pred = frog (0.983)
Initial prob of TRUE class (frog): 0.9831
Attack succeeded: frog → cat (confidence 0.679)

Attacking image 2 | true = automobile | pred = automobile (0.979)
Initial prob of TRUE class (automobile): 0.9786
Attack failed: still automobile (confidence 0.924)

Attacking image 3 | true = cat | pred = cat (0.944)
Initial prob of TRUE class (cat): 0.9441
Attack succeeded: cat → deer (confidence 0.660)

Attacking image 4 | true = dog | pred = dog (0.985)
Initial prob of TRUE class (dog): 0.9847
Attack succeeded: dog → bird (confidence 0.490)

Attacking image 5 | true = frog | pred = frog (0.986)
Initial prob of TRUE class (frog): 0.9857
Attack succeeded: frog → deer (confidence 0.466)

Attacking image 6 | true = dog | pred = dog (0.983)
Initial prob of TRUE class (dog): 0.9826
Attack failed: still dog (confidence 0.764)

Attacking image 7 | true = cat | pred = cat (0.763)
Initial prob of TRUE class (cat): 0.7631
At

60.0

In [25]:
# Run 2-pixel attack
evaluate_attack_n_pixels(model, testloader, mean, std, n_pixels=2, num_images=100, popsize=80, maxiter=40)


Running 2-PIXEL ATTACK


Attacking image 1 | true = cat | pred = cat (0.980)
Initial prob of TRUE class (cat): 0.9805
Attack succeeded: cat → dog (confidence 0.565)

Attacking image 2 | true = dog | pred = dog (0.984)
Initial prob of TRUE class (dog): 0.9843
Attack failed: still dog (confidence 0.976)

Attacking image 3 | true = horse | pred = horse (0.986)
Initial prob of TRUE class (horse): 0.9858
Attack succeeded: horse → dog (confidence 0.556)

Attacking image 4 | true = cat | pred = cat (0.941)
Initial prob of TRUE class (cat): 0.9411
Attack succeeded: cat → frog (confidence 0.340)

Attacking image 5 | true = deer | pred = deer (0.990)
Initial prob of TRUE class (deer): 0.9899
Attack succeeded: deer → dog (confidence 0.966)

Attacking image 6 | true = bird | pred = bird (0.990)
Initial prob of TRUE class (bird): 0.9903
Attack succeeded: bird → airplane (confidence 0.939)

Attacking image 7 | true = deer | pred = deer (0.990)
Initial prob of TRUE class (deer): 0.9899
Attack succee

72.0

In [26]:
# Run 3-pixel attack
evaluate_attack_n_pixels(model, testloader, mean, std, n_pixels=3, num_images=100, popsize=80, maxiter=40)


Running 3-PIXEL ATTACK


Attacking image 1 | true = airplane | pred = airplane (0.993)
Initial prob of TRUE class (airplane): 0.9933
Attack failed: still airplane (confidence 0.965)

Attacking image 2 | true = ship | pred = ship (0.992)
Initial prob of TRUE class (ship): 0.9917
Attack succeeded: ship → airplane (confidence 0.480)

Attacking image 3 | true = frog | pred = frog (0.984)
Initial prob of TRUE class (frog): 0.9844
Attack succeeded: frog → bird (confidence 0.650)

Attacking image 4 | true = ship | pred = ship (0.985)
Initial prob of TRUE class (ship): 0.9851
Attack succeeded: ship → airplane (confidence 0.989)

Attacking image 5 | true = bird | pred = bird (0.988)
Initial prob of TRUE class (bird): 0.9876
Attack succeeded: bird → cat (confidence 0.890)

Attacking image 6 | true = deer | pred = deer (0.984)
Initial prob of TRUE class (deer): 0.9839
Attack succeeded: deer → bird (confidence 0.809)

Attacking image 7 | true = bird | pred = bird (0.987)
Initial prob of TRUE clas

82.0

In [12]:
# Test 1-pixel attack: 5 images per class
stats_1pixel = evaluate_attack_per_class(
    model, testloader, mean, std,
    n_pixels=1,
    images_per_class=50,
    popsize=80,
    maxiter=40
)


Per-Class 1-Pixel Attack Evaluation
Testing 50 correctly classified images per class

[horse 1/50] conf: 0.986 Attack failed: still horse (confidence 0.957)
[horse 2/50] conf: 0.986 Attack failed: still horse (confidence 0.976)
[cat 1/50] conf: 0.986 Attack failed: still cat (confidence 0.965)
[horse 3/50] conf: 0.980 Attack succeeded: horse → automobile (confidence 0.600)
[frog 1/50] conf: 0.989 Attack failed: still frog (confidence 0.587)
[ship 1/50] conf: 0.992 Attack succeeded: ship → frog (confidence 0.384)
[frog 2/50] conf: 0.982 Attack succeeded: frog → cat (confidence 0.925)
[deer 1/50] conf: 0.983 Attack failed: still deer (confidence 0.640)
[horse 4/50] conf: 0.986 Attack failed: still horse (confidence 0.978)
[bird 1/50] conf: 0.991 Attack failed: still bird (confidence 0.962)
[automobile 1/50] conf: 0.984 Attack failed: still automobile (confidence 0.973)
[dog 1/50] conf: 0.983 Attack failed: still dog (confidence 0.974)
[ship 2/50] conf: 0.989 Attack succeeded: ship → air

In [ ]:
# Test 2-pixel attack: 10 images per class
stats_2pixel = evaluate_attack_per_class(
    model, testloader, mean, std,
    n_pixels=2,
    images_per_class=10,
    popsize=80,
    maxiter=80
)

In [ ]:
# Test 3-pixel attack: 10 images per class
stats_3pixel = evaluate_attack_per_class(
    model, testloader, mean, std,
    n_pixels=3,
    images_per_class=10,
    popsize=100,
    maxiter=100
)